# LLM中文文本生成系统——分批功能测试

本 Notebook 按功能分批测试。请从上到下运行；每个测试单元均说明运行功能和预期结果。默认路径与 Linux 测试服务器一致，所有产物写入新的时间戳目录，不覆盖既有结果。

## 0. 初始化测试环境

**运行功能：** 设置项目、模型、设备和输出路径，并定义命令执行与结果检查函数。  
**预期结果：** 显示四项路径信息；输出目录成功建立。若使用其他服务器，只需修改下面的预设值。

In [ ]:
from pathlib import Path
from datetime import datetime
import json, os, subprocess, sys

PROJECT_ROOT = Path('/home/wangzy/workplace/wyc/llmctg')
MODEL_PATH = Path('/home/wangzy/workplace/wyc/Qwen3-1.7B')
DEVICE = 'cuda:0'
OUTPUT_ROOT = PROJECT_ROOT / 'test_output' / f"notebook_{datetime.now():%Y%m%d_%H%M%S}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

ENV = os.environ.copy()
ENV['CUDA_VISIBLE_DEVICES'] = DEVICE.split(':', 1)[1] if DEVICE.startswith('cuda:') else ''
PROCESS_DEVICE = 'cuda:0' if DEVICE.startswith('cuda:') else DEVICE

def run_llmctg(arguments, accepted=(0,)):
    command = [sys.executable, '-m', 'llmctg', *map(str, arguments)]
    print('执行命令:', ' '.join(command))
    result = subprocess.run(command, cwd=PROJECT_ROOT, env=ENV, text=True)
    assert result.returncode in accepted, f'退出码 {result.returncode}，预期 {accepted}'

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]

print('项目路径:', PROJECT_ROOT)
print('模型路径:', MODEL_PATH)
print('输出路径:', OUTPUT_ROOT)
print('计算设备:', DEVICE)

## 1. 命令入口与默认配置

**运行功能：** 检查版本命令并生成默认 JSON 配置。  
**预期结果：** 显示 `llmctg 1.0.0`；配置包含 `generation` 和 `evaluation`。

In [ ]:
run_llmctg(['--version'])
config_file = OUTPUT_ROOT / 'llmctg.json'
run_llmctg(['init-config', '--output', config_file])
config = json.loads(config_file.read_text(encoding='utf-8'))
assert {'generation', 'evaluation'} <= config.keys()
print('PASS：命令入口和配置生成功能正常。')

## 2. 训练数据转换

**运行功能：** 将带 TAB 等级标签的文本转换为训练 JSONL。  
**预期结果：** 生成 12 条包含 `instruction`、`input`、`output`、`level` 的记录。

In [ ]:
TRAIN_FILE = OUTPUT_ROOT / 'prepared_train.jsonl'
run_llmctg(['prepare-data', '--input', PROJECT_ROOT / 'test_data/llmctg_smoke_train.txt', '--output', TRAIN_FILE])
train_records = read_jsonl(TRAIN_FILE)
assert len(train_records) == 12
assert {'instruction', 'input', 'output', 'level'} <= train_records[0].keys()
print('PASS：训练数据转换正常，共', len(train_records), '条。')

## 3. 模板生成与文本评估

**运行功能：** 使用无需模型的模板后端生成文章，并对指定中文文本进行质量评估。  
**预期结果：** 生成结果的 `backend` 为 `template`；评估 JSON 包含统计和质量指标。评估命令返回 1 仅表示文本未达到及格线，不代表功能失败。

In [ ]:
template_file = OUTPUT_ROOT / 'template_generation.jsonl'
run_llmctg(['generate', '--backend', 'template', '--topic', '城市河流', '--level', 'junior', '--keywords', '河流,生态,保护', '--count', '1', '--seed', '2026', '--output', template_file])
assert read_jsonl(template_file)[0]['backend'] == 'template'

evaluation_file = OUTPUT_ROOT / 'evaluation.json'
sample_text = '城市河流为植物和动物提供生活空间，也能调节局部气候。保护河流需要减少污水排放、修复河岸植被，并鼓励居民节约用水。'
run_llmctg(['evaluate', '--text', sample_text, '--level', 'junior', '--output', evaluation_file], accepted=(0, 1))
evaluation = json.loads(evaluation_file.read_text(encoding='utf-8'))
assert {'metrics', 'statistics'} <= evaluation.keys()
print('PASS：模板生成与文本评估正常。')

## 4. 批处理与完整演示

**运行功能：** 执行 3 条 JSONL 批量任务，再运行内置完整演示。  
**预期结果：** 两个目录均生成文章、摘要、CSV 和 HTML 报告；各包含 3 篇文章。

In [ ]:
batch_dir = OUTPUT_ROOT / 'batch'
run_llmctg(['batch', '--backend', 'template', '--input', PROJECT_ROOT / 'examples/tasks.jsonl', '--output', batch_dir, '--no-resume'])
assert len(read_jsonl(batch_dir / 'articles.jsonl')) == 3
assert all((batch_dir / name).exists() for name in ['summary.json', 'summary.csv', 'report.html'])

demo_dir = OUTPUT_ROOT / 'demo'
run_llmctg(['demo', '--output', demo_dir])
assert len(read_jsonl(demo_dir / 'articles.jsonl')) == 3
assert (demo_dir / 'report.html').exists()
print('PASS：批处理与完整演示正常。')

## 5. Qwen3 本地基础模型生成

**运行功能：** 在单张 GPU 上加载预设 Qwen3-1.7B 模型并生成中文文章。  
**预期结果：** 输出一条 JSONL，`content` 非空；运行期间仅选定 GPU 被该进程占用。

In [ ]:
base_file = OUTPUT_ROOT / 'qwen3_base_generation.jsonl'
run_llmctg(['generate', '--backend', 'transformers', '--model-path', MODEL_PATH, '--device', PROCESS_DEVICE, '--topic', '城市河流与生态保护', '--level', 'junior', '--instruction', '说明城市河流的生态价值、主要污染来源和保护方法', '--keywords', '河流,生态,污染,保护', '--count', '1', '--seed', '2026', '--output', base_file])
base_records = read_jsonl(base_file)
assert len(base_records) == 1 and base_records[0]['content'].strip()
print('PASS：本地基础模型生成正常。')

## 6. LoRA 冒烟微调

**运行功能：** 使用 12 条样本对 Qwen3-1.7B 进行 1 个 epoch 的 LoRA 微调。  
**预期结果：** 训练正常结束，生成 `adapter_config.json` 和 `adapter_model.safetensors`。单卡预计产生约 3 个优化步；实际数值取决于 Trainer 版本。

In [ ]:
ADAPTER_PATH = OUTPUT_ROOT / 'qwen3-1.7b-lora-smoke'
run_llmctg(['train', '--model-path', MODEL_PATH, '--dataset', TRAIN_FILE, '--output', ADAPTER_PATH, '--epochs', '1', '--batch-size', '1', '--max-length', '256', '--device', PROCESS_DEVICE])
assert (ADAPTER_PATH / 'adapter_config.json').exists()
assert (ADAPTER_PATH / 'adapter_model.safetensors').exists()
print('PASS：LoRA 微调及适配器保存正常。')

## 7. 加载 LoRA 适配器生成

**运行功能：** 使用 `--lora-path` 加载上一单元生成的适配器并进行推理。  
**预期结果：** 成功加载基础模型和 LoRA 权重，输出一条正文非空的 JSONL 记录。

In [ ]:
lora_file = OUTPUT_ROOT / 'qwen3_lora_generation.jsonl'
run_llmctg(['generate', '--backend', 'transformers', '--model-path', MODEL_PATH, '--lora-path', ADAPTER_PATH, '--device', PROCESS_DEVICE, '--topic', '城市河流与生态保护', '--level', 'junior', '--instruction', '说明城市河流的生态价值、主要污染来源和保护方法', '--keywords', '河流,生态,污染,保护', '--count', '1', '--seed', '2026', '--output', lora_file])
lora_records = read_jsonl(lora_file)
assert len(lora_records) == 1 and lora_records[0]['content'].strip()
print('PASS：LoRA 适配器加载与文本生成正常。')
print('全部测试产物：', OUTPUT_ROOT)